# DATA/MSML 641: Natural Language Processing
## Lecture 3: Word Meaning

<div style="display: flex; align-items: center; justify-content: space-between; margin: 40px 0;">
<div style="flex: 1;">

**University of Maryland, College Park**  
**Fall 2025**  
**Instructor**: Armin Mehrabian  
**Date**: September 15-16, 2025

</div>
<div style="flex: 0 0 200px; text-align: right;">

<img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" alt="UMD Logo" style="max-width: 200px; height: auto;" />

</div>
</div>

---

## 🎯 Learning Objectives

By the end of this lecture, you will be able to:
1. Distinguish between different approaches to representing word meaning
2. Understand lexicographic traditions and word sense distinctions
3. Apply decompositional and ontological approaches to semantic analysis
4. Implement word sense disambiguation techniques
5. Evaluate the strengths and limitations of traditional meaning representations

## 📋 Before we begin

**Last lecture recap**: 
- What constitutes a word (hidden complexity!)
- Multi-word expressions (MWE) and collocations
- Statistical hypothesis testing

**Today's focus**:
- **Word** - We've seen the complexity of defining words
- **Meaning** - What do we mean by meaning? 🤔

## 🌍 Why Word Meaning Matters: Real-World Impact

Before diving into technical approaches, let's see how word meaning representations power technologies you use every day:

### **🔍 Search & Discovery**
- **Google Search**: Uses semantic understanding to match "car repairs" with "automotive maintenance"
- **Amazon Product Search**: Finds "wireless headphones" when you search for "bluetooth earbuds"
- **Netflix Recommendations**: Understands that "sci-fi thriller" relates to specific genre preferences

### **🗣️ Language Translation**
- **DeepL vs Google Translate (2024)**: Different approaches to handling ambiguous words
  - Swedish "kön" → "queue" or "gender"? Context determines meaning
  - Technical terms require specialized sense disambiguation

### **🤖 Virtual Assistants**
- **Siri/Alexa**: "Play some jazz" requires understanding music genres
- **Customer Support Bots**: Distinguish between "account balance" (financial) vs "work-life balance" (wellness)

### **📊 Business Intelligence**
- **Social Media Analytics**: Sentiment analysis of product mentions
- **Financial News**: Extracting market-moving information from text
- **Medical Records**: Identifying symptoms and treatments from clinical notes

**Today's Question**: How do we teach machines to understand word meanings as flexibly as humans do?

# 1. Lexicographic Approaches

## The Traditional Dictionary-Making Approach

### The Lexicographic Tradition

- **Lexicography**: The art and science of constructing dictionaries
- **Purpose**: Enumerate word meanings (senses)
- **Key Question**: What are definitions?

**Important insight**: 
> Definitions are not "defining" in the mathematical or legal sense. They are **text you read that evoke concepts in your head**.

**Example**: Legal definition of "assault"
- Intentional, affects another person with reasonable apprehension  
- Harmful or offensive contact, imminently
- Physical injury **NOT** required

### 🏢 **Modern Applications**

**Google's Knowledge Graph** uses lexicographic principles:
- Entities have multiple senses: "Apple" (company) vs "Apple" (fruit)  
- Each sense has structured properties and relationships
- Powers search understanding and featured snippets

**Legal Tech Companies** (e.g., Westlaw, LexisNexis):
- Maintain precise definitions for legal terms
- Context determines which legal sense applies
- Critical for case law research and compliance

### Forms of Word Sense Distinction

#### 1. **Homonymy** 
Different words that happen to have the same spelling/pronunciation

**Example: "Pen"**
- "Place to keep animals" (enclosure)
- "Instrument to write with" (writing tool)
- *Question*: Coincidental or historical relationship?

#### 2. **Polysemy**
Single word with multiple **related** meanings

**Example: "Bass"**
- "Low register" (music)
- "Person who sings low" (singer)
- *Note*: Related meanings, not coincidental

#### 3. **Systematic/Structured Polysemy**
Polysemy that applies systematically to a category

**Example**: Animal ↔ Food alternation
- chicken, lamb, goose, turkey, duck...
- Each has both an **animal sense** and a **food sense**

In [1]:
# Interactive Exercise: Exploring Word Senses
import pandas as pd

# Examples of different types of ambiguity
word_examples = {
    'Word': ['bank', 'bark', 'chicken', 'bass', 'mouse', 'turkey'],
    'Sense 1': ['financial institution', 'dog sound', 'bird', 'low voice/music', 'small animal', 'bird'],
    'Sense 2': ['river edge', 'tree covering', 'meat', 'fish', 'computer device', 'silly person'],
    'Type': ['homonymy', 'homonymy', 'systematic polysemy', 'polysemy', 'polysemy', 'polysemy']
}

df = pd.DataFrame(word_examples)
print("🔍 Word Sense Examples")
print(df.to_string(index=False))

print("\n💭 Discussion Questions:")
print("1. Can you identify which examples show homonymy vs. polysemy?")
print("2. What other animal/food pairs can you think of?")
print("3. How would you determine if two senses are related or coincidental?")

🔍 Word Sense Examples
   Word               Sense 1         Sense 2                Type
   bank financial institution      river edge            homonymy
   bark             dog sound   tree covering            homonymy
chicken                  bird            meat systematic polysemy
   bass       low voice/music            fish            polysemy
  mouse          small animal computer device            polysemy
 turkey                  bird    silly person            polysemy

💭 Discussion Questions:
1. Can you identify which examples show homonymy vs. polysemy?
2. What other animal/food pairs can you think of?
3. How would you determine if two senses are related or coincidental?


### Dictionary-Making Approaches

#### **Labor-Intensive Tradition**
- **Oxford English Dictionary (OED)**: Started 1857
- **Crowdsourcing**: Volunteers found quotations for each word
- **Timeline**: More than 50 years to complete!
- **Reference**: Simon Winchester, *The Professor and the Madman*

#### **Corpus Analysis Approach**
- **Collins COBUILD Dictionary**: Published 1987
- **Method**: KWIC (Keyword in Context) analysis
- **Data-driven**: Based on actual language usage patterns

In [2]:
# Demo: KWIC (Keyword in Context) Analysis
import re

def kwic_analysis(text, target_word, window=3):
    """Extract KWIC concordances for a target word"""
    words = text.lower().split()
    concordances = []
    
    for i, word in enumerate(words):
        if target_word.lower() in word:
            # Extract context window
            start = max(0, i - window)
            end = min(len(words), i + window + 1)
            
            left_context = " ".join(words[start:i])
            right_context = " ".join(words[i+1:end])
            
            concordances.append({
                'left': left_context,
                'keyword': word,
                'right': right_context
            })
    
    return concordances

# Example sentences with "bass"
bass_examples = """
He will sing a bass solo after the intermission.
She has a lovely bass voice that resonates well.
The pianist struggled to reach the bass notes on the lower octave.
The band played bass guitar and drums loudly.
The ensemble featured a bass and a lead guitarist performing together.
We caught several bass while fishing in the lake.
The striped bass is common in these coastal waters.
"""

# Analyze "bass" in context
concordances = kwic_analysis(bass_examples, "bass")

print("🎵 KWIC Analysis: 'bass'")
print("=" * 60)
for i, conc in enumerate(concordances, 1):
    print(f"{i:2d}. {conc['left']:25} {conc['keyword']:8} {conc['right']}")

print("\n💡 Observations:")
print("- Can you identify which uses refer to music vs. fish?")
print("- What contextual clues help disambiguation?")

🎵 KWIC Analysis: 'bass'
 1. will sing a               bass     solo after the
 2. has a lovely              bass     voice that resonates
 3. to reach the              bass     notes on the
 4. the band played           bass     guitar and drums
 5. ensemble featured a       bass     and a lead
 6. we caught several         bass     while fishing in
 7. lake. the striped         bass     is common in

💡 Observations:
- Can you identify which uses refer to music vs. fish?
- What contextual clues help disambiguation?


# 2. Decompositional Approaches

## Breaking Down Meaning into Components

### Lexical Decomposition Theory

**Core assumption**: Lexical meaning is complex and can be expressed in structured meaning representations

**Example sentence**: "Jill likes Jack most of the time"
- **Predicates**: likes
- **Arguments**: Jill & Jack  
- **Operators**: "what", "most of the time"

### Approach 1: Necessary and Sufficient Conditions

**Traditional approach** (Katz and Fodor): Definitions provide necessary and sufficient conditions

**Example**: "bachelor" iff {human, male, adult, never married}

#### **Challenge**: Very hard to identify necessary and sufficient conditions!

**Question**: Is having four legs a necessary condition for being a dog?  
→ Does a dog that lost a leg in an accident no longer qualify as a dog? 🤔

In [3]:
# Interactive Exercise: Necessary vs. Sufficient Conditions

def analyze_concept_conditions(concept, proposed_features):
    """Analyze whether features are necessary/sufficient for a concept"""
    print(f"🔍 Analyzing concept: '{concept}'")
    print(f"Proposed features: {proposed_features}")
    
    print("\n🤔 Consider these edge cases:")
    return proposed_features

# Test traditional definitions
concepts = {
    'dog': ['animal', 'four legs', 'barks', 'domesticated'],
    'bird': ['animal', 'has wings', 'can fly', 'has feathers'],
    'chair': ['furniture', 'has legs', 'for sitting', 'has back']
}

for concept, features in concepts.items():
    analyze_concept_conditions(concept, features)
    print("\n" + "="*50 + "\n")

print("💭 Discussion:")
print("- What about a three-legged dog? Still a dog?")
print("- What about penguins? Birds that can't fly?")
print("- What about a stool? Chair without a back?")
print("- How do fuzzy boundaries affect computational systems?")

🔍 Analyzing concept: 'dog'
Proposed features: ['animal', 'four legs', 'barks', 'domesticated']

🤔 Consider these edge cases:


🔍 Analyzing concept: 'bird'
Proposed features: ['animal', 'has wings', 'can fly', 'has feathers']

🤔 Consider these edge cases:


🔍 Analyzing concept: 'chair'
Proposed features: ['furniture', 'has legs', 'for sitting', 'has back']

🤔 Consider these edge cases:


💭 Discussion:
- What about a three-legged dog? Still a dog?
- What about penguins? Birds that can't fly?
- What about a stool? Chair without a back?
- How do fuzzy boundaries affect computational systems?


### Approach 2: Structured Conceptual Representations

**Jackendoff's Conceptual Structure Theory**:

> "Conceptual structure is not a part of language per se – it is a part of **thought**. It is the locus for understanding linguistic utterances in context, incorporating pragmatic considerations and 'world knowledge'."

### Lexical-Conceptual Structure (LCS)

**Semantic primitives**: CAUSE, GO, TO, FROM, etc.

**Example: "give"**
```
give: [CAUSE (x, [GO (y, [TO (z)])])]
```

**"John gave me the book"** →  
John (=X) CAUSED the book (=Y) to GO TO me (=Z)

In [4]:
# Demo: Lexical Conceptual Structure Analysis

class LCSAnalyzer:
    """Simple LCS representation for verbs"""
    
    def __init__(self):
        self.verb_templates = {
            'give': 'CAUSE(X, GO(Y, TO(Z)))',
            'take': 'CAUSE(X, GO(Y, FROM(Z), TO(X)))',
            'put': 'CAUSE(X, GO(Y, TO(Z)))',
            'remove': 'CAUSE(X, GO(Y, FROM(Z)))',
            'send': 'CAUSE(X, GO(Y, TO(Z)))',
            'bring': 'CAUSE(X, GO(Y, TO(HERE)))'
        }
    
    def analyze_sentence(self, sentence, verb, arguments):
        """Analyze a sentence using LCS representation"""
        if verb.lower() in self.verb_templates:
            template = self.verb_templates[verb.lower()]
            print(f"📝 Sentence: {sentence}")
            print(f"🔧 Verb: {verb}")
            print(f"📊 LCS Template: {template}")
            
            if len(arguments) >= 3:
                x, y, z = arguments[0], arguments[1], arguments[2]
                print(f"\n🎯 Instantiated:")
                instantiated = template.replace('X', x).replace('Y', y).replace('Z', z)
                print(f"   {instantiated}")
                
                print(f"\n🔄 Interpretation:")
                print(f"   {x} CAUSED {y} to GO TO {z}")
            
        else:
            print(f"❌ No LCS template available for '{verb}'")
    
    def compare_verbs(self, verbs):
        """Compare LCS structures of multiple verbs"""
        print("🔄 LCS Comparison:")
        for verb in verbs:
            if verb in self.verb_templates:
                print(f"  {verb:8} → {self.verb_templates[verb]}")

# Create analyzer and test
lcs = LCSAnalyzer()

# Analyze example sentences
examples = [
    ("John gave Mary the book", "give", ["John", "the book", "Mary"]),
    ("Sarah took the keys from Tom", "take", ["Sarah", "the keys", "Tom"]),
    ("The robot put the box on the shelf", "put", ["The robot", "the box", "the shelf"])
]

for sentence, verb, args in examples:
    lcs.analyze_sentence(sentence, verb, args)
    print("\n" + "="*60 + "\n")

# Compare related verbs
lcs.compare_verbs(['give', 'take', 'put', 'send', 'bring'])

📝 Sentence: John gave Mary the book
🔧 Verb: give
📊 LCS Template: CAUSE(X, GO(Y, TO(Z)))

🎯 Instantiated:
   CAUSE(John, GO(the book, TO(Mary)))

🔄 Interpretation:
   John CAUSED the book to GO TO Mary


📝 Sentence: Sarah took the keys from Tom
🔧 Verb: take
📊 LCS Template: CAUSE(X, GO(Y, FROM(Z), TO(X)))

🎯 Instantiated:
   CAUSE(Sarah, GO(the keys, FROM(Tom), TO(Sarah)))

🔄 Interpretation:
   Sarah CAUSED the keys to GO TO Tom


📝 Sentence: The robot put the box on the shelf
🔧 Verb: put
📊 LCS Template: CAUSE(X, GO(Y, TO(Z)))

🎯 Instantiated:
   CAUSE(The robot, GO(the box, TO(the shelf)))

🔄 Interpretation:
   The robot CAUSED the box to GO TO the shelf


🔄 LCS Comparison:
  give     → CAUSE(X, GO(Y, TO(Z)))
  take     → CAUSE(X, GO(Y, FROM(Z), TO(X)))
  put      → CAUSE(X, GO(Y, TO(Z)))
  send     → CAUSE(X, GO(Y, TO(Z)))
  bring    → CAUSE(X, GO(Y, TO(HERE)))


# 3. Ontological Approaches

## Organizing Knowledge in Hierarchies

### Ontologies

**Definition**: Sets of entities and relations between them

**Key relationship**: **IS-A** (subsumption) for concepts

**Examples**:
- Biological taxonomies (most familiar)
- Gene ontologies
- Business ontologies  
- Astronomy ontologies

### IS-A Inheritance

**Core principle**: C2 IS-A C1 ⟹ for all properties f of C1, f is also a property of C2

**Connection**: Related to subclassing in object-oriented programming!

In [5]:
# Demo: Building a Simple Ontology
import networkx as nx
import matplotlib.pyplot as plt

class SimpleOntology:
    """A simple ontology with IS-A relationships"""
    
    def __init__(self):
        self.graph = nx.DiGraph()
        self.properties = {}  # concept -> set of properties
    
    def add_concept(self, concept, properties=None):
        """Add a concept with its properties"""
        self.graph.add_node(concept)
        self.properties[concept] = set(properties or [])
    
    def add_isa_relation(self, child, parent):
        """Add IS-A relation: child IS-A parent"""
        self.graph.add_edge(child, parent, relation="IS-A")
    
    def get_inherited_properties(self, concept):
        """Get all properties of a concept (including inherited ones)"""
        all_properties = set(self.properties.get(concept, []))
        
        # Inherit from parents
        for parent in self.graph.successors(concept):
            parent_props = self.get_inherited_properties(parent)
            all_properties.update(parent_props)
        
        return all_properties
    
    def display_hierarchy(self, root_concept):
        """Display the concept hierarchy"""
        def print_tree(concept, level=0):
            indent = "  " * level
            properties = self.properties.get(concept, set())
            props_str = f" [{', '.join(properties)}]" if properties else ""
            print(f"{indent}{concept}{props_str}")
            
            # Find children (concepts that have this as parent)
            children = [node for node in self.graph.nodes() 
                       if concept in self.graph.successors(node)]
            
            for child in children:
                print_tree(child, level + 1)
        
        print_tree(root_concept)

# Build animal ontology
ontology = SimpleOntology()

# Add concepts with properties
ontology.add_concept("Animal", ["alive", "moves", "reproduces"])
ontology.add_concept("Mammal", ["warm-blooded", "has_fur"])
ontology.add_concept("Fish", ["cold-blooded", "has_scales", "lives_in_water"])
ontology.add_concept("Dog", ["domesticated", "barks"])
ontology.add_concept("Bear", ["large", "omnivore"])
ontology.add_concept("Poodle", ["curly_fur", "intelligent"])
ontology.add_concept("Beagle", ["good_nose", "hunting_dog"])
ontology.add_concept("Remy", ["pet", "brown_fur"])  # instance

# Add IS-A relations
ontology.add_isa_relation("Mammal", "Animal")
ontology.add_isa_relation("Fish", "Animal")
ontology.add_isa_relation("Dog", "Mammal")
ontology.add_isa_relation("Bear", "Mammal")
ontology.add_isa_relation("Poodle", "Dog")
ontology.add_isa_relation("Beagle", "Dog")
ontology.add_isa_relation("Remy", "Beagle")  # INSTANCE-OF

print("🌳 Animal Ontology Hierarchy:")
print("=" * 40)
ontology.display_hierarchy("Animal")

print("\n🔍 Property Inheritance Examples:")
print("=" * 40)
test_concepts = ["Remy", "Poodle", "Dog", "Mammal"]
for concept in test_concepts:
    props = ontology.get_inherited_properties(concept)
    print(f"{concept:8}: {sorted(props)}")

🌳 Animal Ontology Hierarchy:
Animal [alive, reproduces, moves]
  Mammal [has_fur, warm-blooded]
    Dog [barks, domesticated]
      Poodle [curly_fur, intelligent]
      Beagle [good_nose, hunting_dog]
        Remy [brown_fur, pet]
    Bear [large, omnivore]
  Fish [has_scales, lives_in_water, cold-blooded]

🔍 Property Inheritance Examples:
Remy    : ['alive', 'barks', 'brown_fur', 'domesticated', 'good_nose', 'has_fur', 'hunting_dog', 'moves', 'pet', 'reproduces', 'warm-blooded']
Poodle  : ['alive', 'barks', 'curly_fur', 'domesticated', 'has_fur', 'intelligent', 'moves', 'reproduces', 'warm-blooded']
Dog     : ['alive', 'barks', 'domesticated', 'has_fur', 'moves', 'reproduces', 'warm-blooded']
Mammal  : ['alive', 'has_fur', 'moves', 'reproduces', 'warm-blooded']


### WordNet: The Most Famous Ontology

**Created by**: George Miller (mid-1980s, passed away 2012 at age 92)  
**Scope**: Originally English, now multilingual WordNets  
**Organization**: Separate ontologies by part of speech (N, V, Adj, Adv)

#### **Core Concept: Synset** (Synonym Set) ≈ "Concept"
- `⟨board, plank, beam⟩` - wood pieces
- `⟨board, committee, panel, council⟩` - governing bodies

#### **Key Relations**:
- **Hypernym/Hyponym** (IS-A): dog IS-A animal
- **Instance** (INSTANCE-OF): Remy INSTANCE-OF beagle  
- **Meronym** (PART-OF): wheel PART-OF car
- **Antonym** (OPPOSITE): hot OPPOSITE cold

In [6]:
# Demo: Working with WordNet
import nltk
from nltk.corpus import wordnet as wn

# Download WordNet if not already available
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

def explore_wordnet_synsets(word):
    """Explore WordNet synsets for a word"""
    synsets = wn.synsets(word)
    
    print(f"🔍 WordNet Analysis: '{word}'")
    print(f"Found {len(synsets)} synset(s):\n")
    
    for i, synset in enumerate(synsets, 1):
        print(f"{i}. {synset.name()}")
        print(f"   Definition: {synset.definition()}")
        print(f"   Examples: {synset.examples()}")
        print(f"   Lemmas: {[lemma.name() for lemma in synset.lemmas()]}")
        
        # Show hypernyms (IS-A parents)
        hypernyms = synset.hypernyms()
        if hypernyms:
            print(f"   Hypernyms: {[h.name() for h in hypernyms]}")
        
        print()

def compare_word_similarity(word1, word2):
    """Compare semantic similarity using WordNet"""
    synsets1 = wn.synsets(word1)
    synsets2 = wn.synsets(word2)
    
    if not synsets1 or not synsets2:
        print(f"❌ No synsets found for {word1} or {word2}")
        return
    
    # Get most similar pair
    max_similarity = 0
    best_pair = None
    
    for s1 in synsets1:
        for s2 in synsets2:
            similarity = s1.path_similarity(s2)
            if similarity and similarity > max_similarity:
                max_similarity = similarity
                best_pair = (s1, s2)
    
    print(f"🔗 Similarity between '{word1}' and '{word2}': {max_similarity:.3f}")
    if best_pair:
        print(f"   Best match: {best_pair[0].name()} ↔ {best_pair[1].name()}")

# Explore examples
words_to_explore = ['dog', 'bass']
for word in words_to_explore:
    explore_wordnet_synsets(word)
    print("=" * 60)

# Compare word similarities
print("\n🔍 Semantic Similarity Examples:")
word_pairs = [('dog', 'cat'), ('dog', 'animal'), ('car', 'vehicle'), ('bass', 'guitar')]
for w1, w2 in word_pairs:
    compare_word_similarity(w1, w2)

[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1032)>


🔍 WordNet Analysis: 'dog'
Found 8 synset(s):

1. dog.n.01
   Definition: a member of the genus Canis (probably descended from the common wolf) that has been domesticated by man since prehistoric times; occurs in many breeds
   Examples: ['the dog barked all night']
   Lemmas: ['dog', 'domestic_dog', 'Canis_familiaris']
   Hypernyms: ['domestic_animal.n.01', 'canine.n.02']

2. frump.n.01
   Definition: a dull unattractive unpleasant girl or woman
   Examples: ['she got a reputation as a frump', "she's a real dog"]
   Lemmas: ['frump', 'dog']
   Hypernyms: ['unpleasant_woman.n.01']

3. dog.n.03
   Definition: informal term for a man
   Examples: ['you lucky dog']
   Lemmas: ['dog']
   Hypernyms: ['chap.n.01']

4. cad.n.01
   Definition: someone who is morally reprehensible
   Examples: ['you dirty dog']
   Lemmas: ['cad', 'bounder', 'blackguard', 'dog', 'hound', 'heel']
   Hypernyms: ['villain.n.01']

5. frank.n.02
   Definition: a smooth-textured sausage of minced beef or pork usually s

# 4. Word Sense Disambiguation (WSD)

## A Classic Problem in NLP

### WSD as Classification Problem

#### **Problem Formulation**:
- **Given**: 
  - Enumerated senses {s₁, s₂, …, sₙ} for word w
  - Context for w: (…. w ….)
- **Select**: "Correct" sᵢ for w

#### **Simple Approaches**:
- **Strong baseline**: Most frequent sense (surprisingly effective!)
- **Context-based**: Look at surrounding words
- **Statistical methods**: We'll learn these in **Week 8: Machine Learning in NLP**

#### **Key Insight**: Context is crucial for disambiguation
- "The **bank** charged a fee" (financial institution)
- "We sat by the river **bank**" (edge/shore)

The surrounding words provide the clues needed to distinguish senses.

In [7]:
# Simple WSD Example: Context Clues
import pandas as pd

def analyze_word_context(sentences, target_word):
    """Simple analysis of context clues for word disambiguation"""
    
    print(f"🔍 Context Analysis for: '{target_word}'")
    print("=" * 50)
    
    examples = []
    
    for i, sentence in enumerate(sentences, 1):
        # Find the target word and extract context
        words = sentence.lower().split()
        
        for j, word in enumerate(words):
            if target_word.lower() in word:
                # Get surrounding words
                left_context = words[max(0, j-2):j]
                right_context = words[j+1:min(len(words), j+3)]
                
                examples.append({
                    'Sentence': sentence,
                    'Left Context': ' '.join(left_context),
                    'Target': word,
                    'Right Context': ' '.join(right_context),
                    'Likely Sense': ''  # Students can fill this in
                })
    
    # Display as table
    df = pd.DataFrame(examples)
    print(df.to_string(index=False))
    
    return examples

# Example sentences with "bass"
bass_sentences = [
    "She plays bass guitar in the jazz band.",
    "Her bass voice resonated through the auditorium.",
    "We caught several bass while fishing at the lake.",
    "The bass notes were difficult to reach on the piano.",
    "Striped bass are common in these coastal waters.",
    "The choir needs more singers in the bass section."
]

# Analyze context clues
examples = analyze_word_context(bass_sentences, "bass")

print("\n🤔 Discussion Questions:")
print("1. What context clues help you identify each sense?")
print("2. Which examples are easiest/hardest to disambiguate?") 
print("3. How would you teach a computer to use these clues?")
print("4. What happens when context clues are ambiguous or missing?")

🔍 Context Analysis for: 'bass'
                                            Sentence   Left Context Target   Right Context Likely Sense
             She plays bass guitar in the jazz band.      she plays   bass       guitar in             
    Her bass voice resonated through the auditorium.            her   bass voice resonated             
   We caught several bass while fishing at the lake. caught several   bass   while fishing             
The bass notes were difficult to reach on the piano.            the   bass      notes were             
    Striped bass are common in these coastal waters.        striped   bass      are common             
   The choir needs more singers in the bass section.         in the   bass        section.             

🤔 Discussion Questions:
1. What context clues help you identify each sense?
2. Which examples are easiest/hardest to disambiguate?
3. How would you teach a computer to use these clues?
4. What happens when context clues are ambiguous or mis

### "I don't believe in Word Senses" - Kilgariff (2003)

#### **Kilgariff's Arguments**:
- Advocated for **task-dependent clustering** of corpus instances
- **Schütze**: Early distributional representations can form natural clusters
- **Current trend**: Move away from discrete senses

#### **"You shall know a word by the company it keeps"**

This famous quote by J.R. Firth (1957) hints at an alternative approach to word meaning that we'll explore in detail in **Week 9: Vector Semantics & Embeddings**.

### Why doesn't traditional WSD help many applications?

1. **Moderate performance** - Even good systems plateau around 80-85%
2. **Sense skew** - Most frequent sense dominates (Zipf's law)
3. **One sense per discourse** - Context usually disambiguates
4. **Implicit disambiguation** - Co-occurrence provides clues
   - Example: "bank" → ambiguous, but "bank ATM hours" → unambiguous
5. **Evaluation challenges** - Inter-annotator agreement often low

### **Real-World WSD Challenges**

**Translation Systems** still struggle with:
- **Context-dependent ambiguity**: Swedish "kön" → "queue" or "gender"?
- **Technical terminology**: Medical vs. general usage
- **Cultural references**: Idioms and colloquialisms

**This motivates alternative approaches to word meaning we'll cover in later weeks.**

## 🔄 Summary & Key Takeaways

### **Three Approaches to Word Meaning (Week 3)**:

#### **1. Lexicographic** 📖
- **Method**: Traditional dictionary-making
- **Goal**: Enumerate discrete senses with definitions  
- **Strength**: Precise, interpretable
- **Challenge**: Labor-intensive, doesn't scale well
- **Example**: Oxford English Dictionary, WordNet synsets

#### **2. Decompositional** 🔧  
- **Method**: Break meaning into semantic primitives
- **Goal**: Compositional meaning representation
- **Strength**: Systematic, captures relationships
- **Challenge**: Hard to define primitives, coverage gaps
- **Example**: LCS representations like CAUSE(X, GO(Y, TO(Z)))

#### **3. Ontological** 🌳
- **Method**: Hierarchical organization of concepts
- **Goal**: Structured knowledge with inheritance
- **Strength**: Captures relationships, enables reasoning
- **Challenge**: Coverage, agreement on hierarchies
- **Example**: WordNet taxonomy, IS-A relationships

### **Word Sense Disambiguation**
- **Problem**: Given context, choose the right sense
- **Approaches**: Most frequent sense, context analysis
- **Challenge**: Low inter-annotator agreement
- **Reality**: Many applications work around WSD rather than solving it

### **Looking Ahead**
- **Week 4**: How do sequences of words create meaning?
- **Week 8**: Machine learning approaches to these problems  
- **Week 9**: Alternative approaches using distributional semantics

## 📚 Next Week Preview

**Topic**: Sequential Structure in Language  
**Focus**: Hidden Markov Models, sequence labeling, and part-of-speech tagging  
**Reading**: SLP Chapters 3, 8 (through 8.4), 9, Appendices A & B

### Prepare for next week:
- Review probability and Bayes' rule
- Think about how words depend on surrounding context
- Consider: How do we model sequences in language?

---

## 🤔 Discussion Questions

1. **Trade-offs**: What are the advantages and disadvantages of each approach to word meaning?

2. **Modern NLP**: Why have distributional approaches become dominant in modern NLP systems?

3. **Interpretability**: How do we balance performance with interpretability in meaning representations?

4. **Cultural aspects**: How do different approaches handle cultural and contextual variations in meaning?

5. **Future directions**: What might be the next breakthrough in computational semantics?